# 📈 Getting Constant Maturity Yields From FRED
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-One/blob/master/colab/Colab_Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-One/master?urlpath=lab/tree/notebooks/Getting_Constant_Maturity_Yields_From_FRED.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-One/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>
</div>

This notebook puts our secure key management and data pipeline to FRED to work. Before we dive in, you need to have some familiarity with FRED series IDs and how to extract them. As you will see, the `get_series` method of `FredReader` deftly handles incorrect series IDs and can be used to sort out the correct ones.

Start off by watching this short video on the FRED database and its series IDs


[![Getting Series IDs From Fred](https://img.youtube.com/vi/ZP1hKFRZAz4/0.jpg)](https://youtu.be/ZP1hKFRZAz4)

:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::

:::{important} 🛠️ Notebook Setup: Why the "Try/Except" Imports?
:class: dropdown

**The Goal:**
To ensure this notebook runs perfectly whether you are using **Google Colab**, a local **Jupyter instance**, or a remote server without you having to manually install software.

* **External Libraries:** NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.
* **The `try/except` Logic:** This is a safety net.
    1. We **try** to import the library.
    2. If it fails (because it's not installed), the **except** block triggers a `!pip install` to download it automatically.
* **Aliasing (`as np`):** We rename `numpy` to `np` to save keystrokes. In professional finance code, `np` and `pd` are the universal shorthand.
  
:::

## Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported. (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd
```

In [1]:
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

:::{important} ☁️ Cloud-Loading: How In-Memory Modules Work
:class: dropdown

**The Logic:**
Usually, Python looks for modules as `.py` files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.

**The Workflow:**
1. **Fetch:** `requests.get(url)` grabs the raw text of your Python script from Dropbox.
2. **Instantiate:** `ModuleType(module_name)` creates an empty "container" in your computer's RAM.
3. **Execute:** `exec(code, module.__dict__)` runs that text inside the container, turning text into live functions.
4. **Register:** By adding it to `sys.modules`, we tell Python: *"If I try to import this later, don't look on the disk—look right here in the memory."*

**Why do this?**
It makes your notebooks **100% portable**. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."
:::

### Adding a custom module and importing functions


Similar to Chapter One, this notebook utilizes the custom module, **module_basic_concepts_fixed_income**, sourced from Dropbox and named `basic_concepts_fixed_income`. As a reminder, the module is accessible in the notebook's memory, but is not added to a drive.  

The function `secure_key_setup` (**Environment-Aware Secure Key Management**) is imported.  In addition the new class `FredReader` is included in the function.  It isn't imported but is an attirubte of the module that is assigned in the notebook.


```
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (secure_key_setup)
    FredReader=module.FredReader
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")

```



In [2]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (secure_key_setup)
    FredReader=module.FredReader
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")


## ✅ Authenicate your FRED API key

In [3]:
secure_key_setup("fred_key")

## 🏦 🆔 Series IDs for constant maturity yields and the secured overnight funding rate SOFR

Generate a potential list of Series IDs for constant maturity yields as `f` strings..

*   **Monthly Series**: all monthly maturities between one and eleven months

```
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
```


*   **Yearly Series**: all years between one and thirty years

```
yearly_ids = [f"DGS{i}" for i in range(1, 31)]
```

Secured overnight Id is 'SOFR'.  The list of all IDs is series_id.

In [7]:
# Generate months 1-11 and years 1-30 programmatically
monthly_ids = [f"DGS{i}MO" for i in range(1, 12)]
yearly_ids = [f"DGS{i}" for i in range(1, 31)]

# Combine everything together with SOFR
series_ids = ['sofr'] + monthly_ids + yearly_ids

## 🔗 Accessing constant maturity yields for May 2026 with `FredReader`.



*   **Create an instance of the FredRreder**
      ```
      fred_data=FredReader()
      ```
*   **FredReader method `get_series`**
    * **`series_id`** is required and be a string or an iterable of strings.
    * **`startng_date` and `end_date`** are optional.
    * **`ttl`** time for cache to live defaults to 7 days.   
      ```
      yield_data=fred_data.get_series(series_ids,start_date='2026-05-01')
      ```



In [8]:
# create an instance of FredReader
fred_data=FredReader()

# call the method for the class
yield_data=fred_data.get_series(series_ids,start_date='2026-05-01')

✅ Key loaded from local environment ('fred_key')

☁️--- Processing sofr ---
🕒 Metadata is fresh (1 days old).
✅ Loaded sofr from local cache.

☁️--- Processing DGS1MO ---
🕒 Metadata is fresh (1 days old).
✅ Loaded DGS1MO from local cache.

☁️--- Processing DGS2MO ---
🆕 First run for DGS2MO. Initializing metadata...
❌ API REJECTED for 'DGS2MO': Bad Request.  The series does not exist.
⚠️ Skipping DGS2MO: No data was returned.

☁️--- Processing DGS3MO ---
🕒 Metadata is fresh (1 days old).
✅ Loaded DGS3MO from local cache.

☁️--- Processing DGS4MO ---
🆕 First run for DGS4MO. Initializing metadata...
❌ API REJECTED for 'DGS4MO': Bad Request.  The series does not exist.
⚠️ Skipping DGS4MO: No data was returned.

☁️--- Processing DGS5MO ---
🆕 First run for DGS5MO. Initializing metadata...
❌ API REJECTED for 'DGS5MO': Bad Request.  The series does not exist.
⚠️ Skipping DGS5MO: No data was returned.

☁️--- Processing DGS6MO ---
🕒 Metadata is fresh (1 days old).
✅ Loaded DGS6MO from local cach

In [9]:
display(yield_data)

,sofr,DGS1MO,DGS3MO,DGS6MO,DGS1,DGS2,DGS3,DGS5,DGS7,DGS10,DGS20,DGS30
DATE,,,,,,,,,,,,
2026-05-01,3.64,3.71,3.68,3.71,3.73,3.88,3.91,4.02,4.20,4.39,4.96,4.97
2026-05-04,3.63,3.71,3.70,3.76,3.78,3.95,3.98,4.08,4.26,4.45,5.01,5.02
2026-05-05,3.62,3.70,3.69,3.75,3.77,3.93,3.97,4.08,4.25,4.43,4.98,4.98
2026-05-06,3.61,3.70,3.69,3.74,3.73,3.87,3.89,3.99,4.17,4.36,4.92,4.94
2026-05-07,3.60,3.72,3.69,3.74,3.76,3.92,3.94,4.04,4.22,4.41,4.96,4.97
2026-05-08,3.60,3.71,3.69,3.74,3.75,3.90,3.92,4.02,4.19,4.38,4.93,4.95
2026-05-11,3.60,3.71,3.70,3.77,3.79,3.95,3.96,4.07,4.24,4.42,4.97,4.98
2026-05-12,3.60,3.71,3.70,3.77,3.80,4.00,4.01,4.12,4.29,4.46,5.02,5.03
2026-05-13,3.59,3.71,3.69,3.77,3.79,3.98,4.00,4.12,4.28,4.46,5.03,5.03




:::::{admonition} ✍️ FRED Data Challenge


*   Access Overnight Secured Funding Rate And Par Yield For Thirty Year Maturity betweenIn January 1, 2026 and May 20, 2026.
*   Did you access series from cache or FRED?

:::{tip}
Use the first and last column heads of `yield_data` as series IDs.
:::
:::{dropdown} ✅ Example Of Solution
## Example of Code


```py
 # assuming imports of this notebook 
 fred_data=FredReader()
 series_ids=[yield_data.columns[0],yield_data.columns[-1]]
 fred_data.get_series(series_ids,start_date='2026-01-01',end_date='2026-05-20')
```
:::

:::{dropdown} 🎯 Answer
The previous request for FRED data specified May 1, 2026 as the start date. If that was your only request or your last request was seven days ago,the cache for the two series `sofr` and `DGS30` is updated with an API request to FRED.

:::::

